# D-MTHD tweet benchmark: full run

Before running: Settings (right panel) -> Accelerator **GPU T4 x2**, Internet **On**; Add Input -> dataset `andrewmvd/cyberbullying-classification`.

Then **Save Version -> Save & Run All (Commit)** so it keeps running for up to 12 hours after you close the tab. Every stage skips work whose results already exist, so re-running a killed notebook resumes.

Split work across accounts by editing `STUDENTS` in the second cell, for example one account runs only `distilbert-base-uncased:distilbert`.
**Second version (comparison grid):** after the first version finishes, add its output as an input, set `RESUME_FROM` in the cell below to that input's path, and Save & Run All again. Everything already finished is skipped; the new teacher, the heterogeneous students, sweeps, robustness and the quantisation table are added.

In [ ]:
import os, subprocess, glob, shutil, zipfile
REPO = "https://github.com/mahdihasanshadi/THESIS.git"
DEST = "/kaggle/working/dmthd-p3"
if not os.path.exists(os.path.join(DEST, "src", "dmthd")):
    r = subprocess.run(["git", "clone", "-q", REPO, DEST])          # works if the repo is public
    if r.returncode != 0:
        shutil.rmtree(DEST, ignore_errors=True)
        z = glob.glob("/kaggle/input/**/dmthd-p3-code.zip", recursive=True)
        tree = glob.glob("/kaggle/input/**/src/dmthd/train_student.py", recursive=True)
        if z:                                                            # zip attached as-is
            zipfile.ZipFile(z[0]).extractall(DEST)
        elif tree:                                                       # Kaggle auto-extracted the zip
            root = os.path.dirname(os.path.dirname(os.path.dirname(tree[0])))
            shutil.copytree(root, DEST)
        else:
            raise SystemExit("clone failed and no code found among the inputs: attach the code dataset")
os.chdir(DEST)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
print(open("README.md").read()[:400])

In [ ]:
import os, subprocess, glob, sys
env = dict(os.environ, ROOT='/kaggle/working', PYTHONPATH='src', GPU='1', SEEDS='1,2,3',
           COMMITTEES='homo,hetero', MODES='ft,skd,uniform,dmthd',
           TEACHERS='bert-large-uncased:bert-large,GroNLP/hateBERT:hatebert,cardiffnlp/twitter-roberta-base-irony:irony',
           STUDENTS='google/bert_uncased_L-4_H-256_A-4:bert-mini,google/bert_uncased_L-4_H-512_A-8:bert-small,distilbert-base-uncased:distilbert')
env['RESUME_FROM'] = ''   # second version: attach the first version's output as an input and put its path here
raw = (glob.glob('/kaggle/input/**/cyberbullying_tweets.csv', recursive=True) or [None])[0]
print('inputs seen:', glob.glob('/kaggle/input/*'), '| csv:', raw, flush=True)
assert raw, 'dataset not attached: Add Input -> Datasets -> andrewmvd/cyberbullying-classification'
cmd = ['python', 'kaggle/run_benchmark.py', '--dataset', 'tweets', '--stage', 'all'] + (['--raw', raw] if raw else [])
print('running:', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, env=env)
if r.returncode != 0:
    raise SystemExit(f'BENCHMARK FAILED with exit code {r.returncode}: scroll up in this log to the first Traceback')
print('BENCHMARK FINISHED')


In [ ]:
# Pack everything worth keeping so it can be downloaded from the notebook output
!cd /kaggle/working && tar czf dmthd_runs.tgz runs cache/tweets/meta.json data/tweets/report.json && ls -la dmthd_runs.tgz
!python -m dmthd.aggregate --runs /kaggle/working/runs/tweets